# Human Development Index (HDI) Prediction System
## Exploratory Data Analysis (EDA)

This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the Human Development Index (HDI) dataset. 
The analysis explores the relationships between major socioeconomic indicators and a country's overall development rating.

### Import Libraries & Set Settings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set theme and plot settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 16,
    'figure.dpi': 120
})

### Load the HDI Dataset

In [ ]:
# Load the dataset generated by train_model.py
df = pd.read_csv("../dataset/hdi_dataset.csv")
df.head()

### Dataset Overview & Summary Statistics

In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nDataset Columns:")
print(df.columns)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nSummary Statistics:")
df.describe()

### 1. Correlation Analysis
Checking how the independent variables are correlated with the target variable (`HDI`) and whether multicollinearity exists between features.

In [ ]:
plt.figure(figsize=(8, 6))
corr_cols = ["Life_Expectancy", "Expected_Years_Schooling", "Mean_Years_Schooling", "GNI_per_Capita", "HDI"]
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5)
plt.title("Correlation Matrix Heatmap")
plt.show()

### 2. Feature Distributions
Visualizing the skewness, variance, and distribution profiles of all socioeconomic indicators and the target HDI.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
features = ["Life_Expectancy", "Expected_Years_Schooling", "Mean_Years_Schooling", "GNI_per_Capita", "HDI"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for i, (col, color) in enumerate(zip(features, colors)):
    sns.histplot(df[col], kde=True, ax=axes[i], color=color, edgecolor="black", alpha=0.7)
    axes[i].set_title(f"Distribution of {col.replace('_', ' ')}")
    axes[i].set_xlabel("")

axes[5].set_visible(False) # Hide empty subplot
plt.suptitle("Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()

### 3. Scatter Plots (Indicators vs. HDI Target)
Analyzing how each individual socioeconomic dimension influences the overall HDI score. Note the log-linear behavior of GNI per Capita.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.ravel()
scatter_features = ["Life_Expectancy", "Expected_Years_Schooling", "Mean_Years_Schooling", "GNI_per_Capita"]

for i, col in enumerate(scatter_features):
    if col == "GNI_per_Capita":
        # Fit logarithmic scale for GNI
        sns.regplot(data=df, x=col, y="HDI", ax=axes[i], scatter_kws={'alpha':0.5, 'color':'#2b5c8f'}, line_kws={'color':'#d9534f'})
        axes[i].set_xscale('log')
        axes[i].set_xlabel(f"GNI per Capita (Log Scale, PPP $)")
    else:
        sns.regplot(data=df, x=col, y="HDI", ax=axes[i], scatter_kws={'alpha':0.5, 'color':'#2b5c8f'}, line_kws={'color':'#d9534f'})
        axes[i].set_xlabel(col.replace('_', ' '))
    axes[i].set_ylabel("HDI")
    axes[i].set_title(f"{col.replace('_', ' ')} vs HDI")

plt.suptitle("Socioeconomic Indicators vs. HDI", fontsize=16)
plt.tight_layout()
plt.show()

### 4. Boxplots & Strip plots (HDI per Tier)
Analyzing the distribution range of the target index within each development category tier.

In [ ]:
plt.figure(figsize=(9, 5.5))
tier_order = ["Low", "Medium", "High", "Very High"]
sns.boxplot(data=df, x="Development_Tier", y="HDI", order=tier_order, palette="Set2", width=0.4, showfliers=False)
sns.stripplot(data=df, x="Development_Tier", y="HDI", order=tier_order, color="black", alpha=0.3, jitter=0.15, size=4.5)
plt.title("HDI Distribution by Development Tier")
plt.xlabel("Development Tier")
plt.ylabel("Human Development Index (HDI)")
plt.show()

### Summary of Findings
1. **Strong Linearities**: Education indicators (Mean Years and Expected Years of Schooling) and Health (Life Expectancy) exhibit a direct, highly linear relationship with HDI.
2. **Log-Linear GNI per Capita**: GNI per capita demonstrates a logarithmic relationship with HDI, validating the UN's usage of natural log scaling in computing the income index component.
3. **Minimal Overlap**: The HDI boxplot shows distinct, clear separation across the four development tiers, indicating the indicators are strong discriminators of well-being.